# 11 SIRIUS概率指纹：P/B/T预测头与五折结构簇交叉验证

本Notebook承接`10_SIRIUS概率指纹字典与矩阵提取.ipynb`，完成Development内部验证。

核心原则：

- 只读取公共谱图产生的SIRIUS/CSI:FingerID概率指纹，不使用真实结构或正确分子式作为模型输入。
- 以化合物为统计单位；同一化合物同一离子模式的多个feature先取逐位中位数。
- 正负模式通过`absoluteIndex`对齐后再融合，不能直接假定两种模式的列顺序相同。
- 严格沿用冻结的Morgan结构簇五折；每个外折内部再选择超参数，避免OOF泄漏。
- ChemVista Novel/Transfer不在本Notebook中读取或评价，保持外部验证锁定。
- 单独报告SIRIUS成功子集的性能与实际覆盖率。

正式运行建议保持`QUICK_TEST=False`。第一次检查环境时可临时设为`True`。

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
N_OUTER_FOLDS = 5
N_BOOTSTRAP = 1000
N_JOBS = min(8, max(1, (os.cpu_count() or 2) - 1))  # Windows下限制并行，降低内存峰值
QUICK_TEST = False             # 正式结果必须为False
INCLUDE_EXTRA_TREES = True     # 只在主融合路线参加内层选模
OVERWRITE_OUTPUTS = True

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'outputs' / 'pbt_msms_v1').exists() and (candidate / '建模分析').exists():
            return candidate
    raise FileNotFoundError('未找到项目根目录，请从项目目录或建模分析目录启动Notebook。')

ROOT = find_project_root(Path.cwd())
SIRIUS_INPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'sirius_v1' / 'development_inputs_v1'
SIRIUS_RESULT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'sirius_v1' / 'development_results_v1'
DEVELOPMENT_PATH = ROOT / 'outputs' / 'pbt_msms_v1' / 'split_v1' / 'development_compounds.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'sirius_heads_v1'
MODEL_DIR = OUTPUT_DIR / 'fold_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_DIR / 'sirius_head_run_manifest.json'
if manifest_path.exists() and not OVERWRITE_OUTPUTS:
    raise FileExistsError(f'正式输出已存在：{manifest_path}。确认后将OVERWRITE_OUTPUTS改为True。')

TASKS = {
    'P_screen_not_ready_binary': {'column': 'label__P_screen_not_ready_binary', 'kind': 'classification'},
    'B_LOGBCF_REGRESSION': {'column': 'label__B_LOGBCF_REGRESSION', 'kind': 'regression'},
    'T_EC50_MG_L_REGRESSION': {'column': 'label__T_EC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_LC50_MG_L_REGRESSION': {'column': 'label__T_LC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_NOEC_MG_L_REGRESSION': {'column': 'label__T_NOEC_MG_L_REGRESSION', 'kind': 'regression'},
}
ROUTES_TO_MODEL = ['positive_only', 'negative_only', 'available_mode_mean']
if QUICK_TEST:
    N_BOOTSTRAP = 100
    INCLUDE_EXTRA_TREES = False
    ROUTES_TO_MODEL = ['available_mode_mean']

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('正式模式:', not QUICK_TEST, '| CPU并行数:', N_JOBS)

## 1. 读取并审计SIRIUS结果

本模块会验证矩阵行、feature清单、成功状态、5000位字典及正负模式`absoluteIndex`集合。任何关键不一致都会立即停止。

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def require_columns(frame: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f'{name}缺少字段：{missing}')

required_files = {
    'feature_manifest': SIRIUS_INPUT_DIR / 'development_sirius_feature_manifest.csv.gz',
    'positive_rows': SIRIUS_RESULT_DIR / 'positive_fingerprint_rows.csv.gz',
    'negative_rows': SIRIUS_RESULT_DIR / 'negative_fingerprint_rows.csv.gz',
    'positive_matrix': SIRIUS_RESULT_DIR / 'positive_fingerprint_probabilities.npy',
    'negative_matrix': SIRIUS_RESULT_DIR / 'negative_fingerprint_probabilities.npy',
    'positive_dictionary': SIRIUS_RESULT_DIR / 'fingerprint_dictionary_positive.csv',
    'negative_dictionary': SIRIUS_RESULT_DIR / 'fingerprint_dictionary_negative.csv',
    'result_qc': SIRIUS_RESULT_DIR / 'development_sirius_result_qc_checks.csv',
    'development': DEVELOPMENT_PATH,
}
missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('缺少上一步正式输出：\n' + '\n'.join(missing_files))

result_qc = pd.read_csv(required_files['result_qc'])
status_col = 'status' if 'status' in result_qc.columns else None
if status_col and result_qc[status_col].astype(str).str.upper().eq('FAIL').any():
    raise ValueError('SIRIUS结果QC存在FAIL，请先修复Notebook 10输出。')

feature_manifest = pd.read_csv(required_files['feature_manifest'])
development = pd.read_csv(required_files['development'])
rows = {mode: pd.read_csv(required_files[f'{mode}_rows']) for mode in ('positive', 'negative')}
matrices = {mode: np.load(required_files[f'{mode}_matrix'], mmap_mode='r') for mode in ('positive', 'negative')}
dictionaries = {mode: pd.read_csv(required_files[f'{mode}_dictionary']) for mode in ('positive', 'negative')}

require_columns(feature_manifest, ['sirius_feature_id', 'inchikey_2d', 'ion_mode', 'cv_group', 'cv_fold', 'supervised_compound_role'], 'feature_manifest')
require_columns(development, ['inchikey_2d', 'cv_group', 'cv_fold', *[v['column'] for v in TASKS.values()]], 'development')
for mode in ('positive', 'negative'):
    require_columns(rows[mode], ['matrix_row', 'sirius_feature_id', 'status'], f'{mode}_rows')
    require_columns(dictionaries[mode], ['relativeIndex', 'absoluteIndex', 'description', 'Type'], f'{mode}_dictionary')
    if len(rows[mode]) != matrices[mode].shape[0]:
        raise ValueError(f'{mode}行表与矩阵行数不一致')
    if matrices[mode].shape[1] != len(dictionaries[mode]):
        raise ValueError(f'{mode}字典与矩阵列数不一致')
    expected_rows = np.arange(len(rows[mode]))
    if not np.array_equal(rows[mode]['matrix_row'].to_numpy(dtype=int), expected_rows):
        raise ValueError(f'{mode} matrix_row不是从0开始的连续唯一索引')

abs_pos = set(dictionaries['positive']['absoluteIndex'].astype(int))
abs_neg = set(dictionaries['negative']['absoluteIndex'].astype(int))
if abs_pos != abs_neg:
    raise ValueError(f'正负模式指纹绝对索引集合不一致：positive={len(abs_pos)}, negative={len(abs_neg)}, shared={len(abs_pos & abs_neg)}')
canonical_absolute_index = np.array(sorted(abs_pos), dtype=int)
column_order = {}
for mode in ('positive', 'negative'):
    dictionary = dictionaries[mode].copy()
    if dictionary['relativeIndex'].duplicated().any() or dictionary['absoluteIndex'].duplicated().any():
        raise ValueError(f'{mode}指纹字典索引不唯一')
    relative_by_absolute = dictionary.set_index('absoluteIndex')['relativeIndex']
    column_order[mode] = relative_by_absolute.loc[canonical_absolute_index].to_numpy(dtype=int)

feature_counts = feature_manifest['ion_mode'].astype(str).str.casefold().value_counts().to_dict()
print('Feature清单:', feature_counts)
for mode in ('positive', 'negative'):
    success = rows[mode]['status'].eq('success').to_numpy()
    success_values = np.asarray(matrices[mode][success][:, column_order[mode]], dtype=np.float32)
    if not np.isfinite(success_values).all():
        raise ValueError(f'{mode}成功行含NaN或Inf')
    if success_values.size and (success_values.min() < 0 or success_values.max() > 1):
        raise ValueError(f'{mode}概率超出[0,1]')
    print(f'{mode}: matrix={matrices[mode].shape}, success={success.sum()}/{len(success)}')
print('跨模式共有absoluteIndex:', len(canonical_absolute_index))

## 2. 聚合为化合物级指纹并构建三条路线

- `positive_only`：只使用正模式成功feature。
- `negative_only`：只使用负模式成功feature。
- `available_mode_mean`：先分别做模式内中位数，再对可用模式取均值；只有单一模式时保留该模式，不进行填零。

缺少相应模式的化合物在该路线中记为不可预测，不会以零向量冒充真实指纹。

In [ ]:
development = development.loc[
    development['cv_fold'].notna() & development['inchikey_2d'].notna()
].copy()
development['cv_fold'] = development['cv_fold'].astype(int)
if development['inchikey_2d'].duplicated().any():
    raise ValueError('Development化合物主表inchikey_2d不唯一')
if not set(development['cv_fold'].unique()).issubset(set(range(N_OUTER_FOLDS))):
    raise ValueError('Development包含0-4以外的cv_fold')

label_columns = [spec['column'] for spec in TASKS.values()]
compound_index = development[['inchikey_2d', 'cv_group', 'cv_fold', *label_columns]].copy().reset_index(drop=True)
compound_index.insert(0, 'compound_row', np.arange(len(compound_index), dtype=int))
compound_lookup = pd.Series(compound_index['compound_row'].to_numpy(), index=compound_index['inchikey_2d']).to_dict()
n_compounds = len(compound_index)
n_bits = len(canonical_absolute_index)
route_arrays = {
    'positive_only': np.full((n_compounds, n_bits), np.nan, dtype=np.float32),
    'negative_only': np.full((n_compounds, n_bits), np.nan, dtype=np.float32),
}

feature_audit_parts = []
for mode, ion_label in [('positive', 'Positive'), ('negative', 'Negative')]:
    merged = rows[mode].merge(
        feature_manifest[['sirius_feature_id', 'inchikey_2d', 'ion_mode', 'cv_group', 'cv_fold', 'supervised_compound_role']],
        on='sirius_feature_id', how='left', validate='one_to_one', indicator=True,
    )
    if not merged['_merge'].eq('both').all():
        raise ValueError(f'{mode}结果存在无法映射回输入manifest的feature')
    if not merged['ion_mode'].astype(str).str.casefold().eq(mode).all():
        raise ValueError(f'{mode}结果与manifest离子模式不一致')
    if not merged['supervised_compound_role'].eq('development').all():
        raise ValueError(f'{mode}输入混入非Development化合物')
    merged['mode'] = mode
    feature_audit_parts.append(merged[['sirius_feature_id', 'inchikey_2d', 'mode', 'status', 'matrix_row', 'cv_group', 'cv_fold']])
    successful = merged.loc[merged['status'].eq('success')].copy()
    for inchikey, group in successful.groupby('inchikey_2d', sort=False):
        if inchikey not in compound_lookup:
            raise ValueError(f'SIRIUS成功feature不在Development主表：{inchikey}')
        source_rows = group['matrix_row'].to_numpy(dtype=int)
        vectors = np.asarray(matrices[mode][source_rows][:, column_order[mode]], dtype=np.float32)
        route_arrays[f'{mode}_only'][compound_lookup[inchikey]] = np.median(vectors, axis=0).astype(np.float32)

feature_audit = pd.concat(feature_audit_parts, ignore_index=True)
for mode in ('positive', 'negative'):
    total_counts = feature_audit.loc[feature_audit['mode'].eq(mode)].groupby('inchikey_2d').size()
    success_counts = feature_audit.loc[feature_audit['mode'].eq(mode) & feature_audit['status'].eq('success')].groupby('inchikey_2d').size()
    compound_index[f'n_{mode}_features_total'] = compound_index['inchikey_2d'].map(total_counts).fillna(0).astype(int)
    compound_index[f'n_{mode}_features_success'] = compound_index['inchikey_2d'].map(success_counts).fillna(0).astype(int)
    compound_index[f'has_{mode}_only'] = np.isfinite(route_arrays[f'{mode}_only']).all(axis=1)

pos = route_arrays['positive_only']
neg = route_arrays['negative_only']
has_pos = np.isfinite(pos).all(axis=1)
has_neg = np.isfinite(neg).all(axis=1)
fusion = np.full_like(pos, np.nan)
fusion[has_pos & ~has_neg] = pos[has_pos & ~has_neg]
fusion[has_neg & ~has_pos] = neg[has_neg & ~has_pos]
fusion[has_pos & has_neg] = (pos[has_pos & has_neg] + neg[has_pos & has_neg]) / 2.0
route_arrays['available_mode_mean'] = fusion
compound_index['has_available_mode_mean'] = np.isfinite(fusion).all(axis=1)
compound_index['n_available_modes'] = has_pos.astype(int) + has_neg.astype(int)

np.savez_compressed(OUTPUT_DIR / 'sirius_compound_fingerprint_routes.npz', **route_arrays, absoluteIndex=canonical_absolute_index)
compound_index.to_csv(OUTPUT_DIR / 'sirius_compound_fingerprint_index.csv', index=False)
feature_audit.to_csv(OUTPUT_DIR / 'sirius_feature_to_compound_status.csv.gz', index=False, compression='gzip')

coverage_records = []
for route in ROUTES_TO_MODEL:
    available = compound_index[f'has_{route}'].to_numpy(dtype=bool)
    for task, spec in TASKS.items():
        labelled = pd.to_numeric(compound_index[spec['column']], errors='coerce').notna().to_numpy()
        for fold_value in ['all', *range(N_OUTER_FOLDS)]:
            fold_mask = np.ones(n_compounds, dtype=bool) if fold_value == 'all' else compound_index['cv_fold'].eq(fold_value).to_numpy()
            denominator = int((labelled & fold_mask).sum())
            numerator = int((labelled & available & fold_mask).sum())
            coverage_records.append({'representation_route': route, 'task': task, 'fold': fold_value, 'n_labelled': denominator, 'n_predictable': numerator, 'coverage': numerator / denominator if denominator else np.nan})
coverage = pd.DataFrame(coverage_records)
coverage.to_csv(OUTPUT_DIR / 'sirius_head_task_coverage.csv', index=False)
display(compound_index[['has_positive_only', 'has_negative_only', 'has_available_mode_mean', 'n_available_modes']].agg(['sum', 'mean']))
display(coverage.loc[coverage['fold'].eq('all')])

## 3. 嵌套五折模型函数

主融合路线的候选包括L2线性模型和ExtraTrees；正/负单模式路线使用L2线性模型作敏感性分析。回归内层主指标为负MAE，分类内层主指标为PR-AUC。

In [ ]:
def candidate_grid(kind: str, route: str) -> list[dict]:
    if kind == 'classification':
        linear_values = [0.3] if QUICK_TEST else [0.03, 0.3, 3.0, 30.0]
        candidates = [{'family': 'linear', 'C': value} for value in linear_values]
    else:
        linear_values = [10.0] if QUICK_TEST else [0.1, 1.0, 10.0, 100.0]
        candidates = [{'family': 'linear', 'alpha': value} for value in linear_values]
    if INCLUDE_EXTRA_TREES and route == 'available_mode_mean':
        candidates.extend([
            {'family': 'extra_trees', 'max_features': 'sqrt', 'min_samples_leaf': 2},
            {'family': 'extra_trees', 'max_features': 0.1, 'min_samples_leaf': 5},
        ])
    return candidates

def make_model(kind: str, params: dict, seed: int):
    if params['family'] == 'linear':
        estimator = (
            LogisticRegression(C=float(params['C']), solver='liblinear', max_iter=5000, random_state=seed)
            if kind == 'classification' else Ridge(alpha=float(params['alpha']))
        )
        return Pipeline([('scale', StandardScaler()), ('model', estimator)])
    common = dict(
        n_estimators=200 if QUICK_TEST else 350,
        max_features=params['max_features'], min_samples_leaf=int(params['min_samples_leaf']),
        n_jobs=N_JOBS, random_state=seed, bootstrap=False,
    )
    return ExtraTreesClassifier(**common) if kind == 'classification' else ExtraTreesRegressor(**common)

def predict_model(model, kind: str, x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if kind == 'classification':
        probability = model.predict_proba(x)[:, 1].astype(float)
        return (probability >= 0.5).astype(float), probability
    prediction = np.asarray(model.predict(x), dtype=float)
    return prediction, np.full(len(prediction), np.nan)

def primary_score(kind: str, y_true: np.ndarray, prediction: np.ndarray, probability: np.ndarray) -> float:
    if kind == 'classification':
        return float(average_precision_score(y_true, probability)) if np.unique(y_true).size == 2 else np.nan
    return -float(mean_absolute_error(y_true, prediction))

def stable_seed(*parts) -> int:
    text = '|'.join(map(str, parts)).encode('utf-8')
    return int(hashlib.sha256(text).hexdigest()[:8], 16)

def select_candidate(x_train, y_train, train_folds, kind: str, task: str, route: str, outer_fold: int):
    records = []
    candidates = candidate_grid(kind, route)
    for candidate_id, params in enumerate(candidates):
        fold_scores = []
        for inner_val_fold in sorted(np.unique(train_folds)):
            fit_mask = train_folds != inner_val_fold
            val_mask = train_folds == inner_val_fold
            if fit_mask.sum() == 0 or val_mask.sum() == 0:
                continue
            if kind == 'classification' and np.unique(y_train[fit_mask]).size < 2:
                continue
            model = make_model(kind, params, stable_seed(SEED, task, route, outer_fold, candidate_id, inner_val_fold))
            model.fit(x_train[fit_mask], y_train[fit_mask])
            pred, prob = predict_model(model, kind, x_train[val_mask])
            score = primary_score(kind, y_train[val_mask], pred, prob)
            if np.isfinite(score):
                fold_scores.append(score)
            records.append({
                'representation_route': route, 'task': task, 'kind': kind, 'outer_fold': outer_fold,
                'inner_validation_fold': int(inner_val_fold), 'candidate_id': candidate_id,
                'family': params['family'], 'params': json.dumps(params, sort_keys=True), 'inner_score': score,
            })
        mean_score = float(np.mean(fold_scores)) if fold_scores else -np.inf
        for record in records:
            if record['candidate_id'] == candidate_id:
                record['candidate_mean_score'] = mean_score
    summary = pd.DataFrame(records)
    candidate_scores = summary.groupby('candidate_id', as_index=False)['candidate_mean_score'].first()
    if candidate_scores.empty or not np.isfinite(candidate_scores['candidate_mean_score']).any():
        raise RuntimeError(f'无有效内层分数：route={route}, task={task}, outer_fold={outer_fold}')
    best_id = int(candidate_scores.sort_values(['candidate_mean_score', 'candidate_id'], ascending=[False, True]).iloc[0]['candidate_id'])
    return candidates[best_id], float(candidate_scores.loc[candidate_scores['candidate_id'].eq(best_id), 'candidate_mean_score'].iloc[0]), records

print('模型函数就绪；正式候选示例：', candidate_grid('classification', 'available_mode_mean'))

## 4. 运行外层五折并保存折模型

这是耗时最长的代码块。控制台会逐个打印路线、任务和外折进度。输出只包含真正的外折预测；SIRIUS失败或标签缺失的化合物不会被静默填补。

In [ ]:
prediction_parts = []
selection_records = []
inner_records = []
trained_model_paths = []
fold_values = compound_index['cv_fold'].to_numpy(dtype=int)
training_started = time.perf_counter()

for route in ROUTES_TO_MODEL:
    x = route_arrays[route]
    available = np.isfinite(x).all(axis=1)
    for task, spec in TASKS.items():
        y = pd.to_numeric(compound_index[spec['column']], errors='coerce').to_numpy(dtype=float)
        eligible = available & np.isfinite(y)
        for outer_fold in range(N_OUTER_FOLDS):
            train_rows = np.flatnonzero(eligible & (fold_values != outer_fold))
            test_rows = np.flatnonzero(eligible & (fold_values == outer_fold))
            if len(test_rows) == 0:
                raise RuntimeError(f'外折无测试样本：{route}, {task}, fold={outer_fold}')
            if spec['kind'] == 'classification' and np.unique(y[train_rows]).size < 2:
                raise RuntimeError(f'分类训练折只有一个类别：{route}, {task}, fold={outer_fold}')
            print(f'[{route}] {task} fold={outer_fold}: train={len(train_rows)}, test={len(test_rows)}', flush=True)
            best_params, inner_best_score, current_inner = select_candidate(
                x[train_rows], y[train_rows], fold_values[train_rows], spec['kind'], task, route, outer_fold
            )
            inner_records.extend(current_inner)
            model = make_model(spec['kind'], best_params, stable_seed(SEED, 'final', route, task, outer_fold))
            model.fit(x[train_rows], y[train_rows])
            prediction, probability = predict_model(model, spec['kind'], x[test_rows])
            model_path = MODEL_DIR / f'{route}__{task}__fold{outer_fold}.joblib'
            joblib.dump(model, model_path, compress=3)
            trained_model_paths.append(model_path)
            selection_records.append({
                'representation_route': route, 'task': task, 'kind': spec['kind'], 'outer_fold': outer_fold,
                'selected_family': best_params['family'], 'best_params': json.dumps(best_params, sort_keys=True),
                'inner_best_score': inner_best_score, 'n_train': len(train_rows), 'n_test': len(test_rows),
                'model_file': model_path.relative_to(ROOT).as_posix(),
            })
            current = compound_index.iloc[test_rows][['inchikey_2d', 'cv_group', 'cv_fold', 'n_positive_features_success', 'n_negative_features_success', 'n_available_modes']].copy()
            current['task'] = task
            current['kind'] = spec['kind']
            current['representation_route'] = route
            current['model'] = 'nested_selected_primary'
            current['outer_fold'] = outer_fold
            current['selected_family'] = best_params['family']
            current['best_params'] = json.dumps(best_params, sort_keys=True)
            current['inner_best_score'] = inner_best_score
            current['y_true'] = y[test_rows]
            current['y_prediction'] = prediction
            current['y_probability'] = probability
            prediction_parts.append(current)

oof = pd.concat(prediction_parts, ignore_index=True)
selection = pd.DataFrame(selection_records)
inner_scores = pd.DataFrame(inner_records)
oof.to_csv(OUTPUT_DIR / 'sirius_head_compound_level_oof.csv.gz', index=False, compression='gzip')
selection.to_csv(OUTPUT_DIR / 'sirius_head_selected_hyperparameters.csv', index=False)
inner_scores.to_csv(OUTPUT_DIR / 'sirius_head_nested_cv_scores.csv.gz', index=False, compression='gzip')
print(f'训练完成，用时 {(time.perf_counter() - training_started) / 60:.1f} 分钟；OOF行数={len(oof)}')
display(selection.groupby(['representation_route', 'task', 'selected_family']).size().rename('n_outer_folds').reset_index())

## 5. 统一指标、化合物级bootstrap置信区间与P校准

P主指标为PR-AUC；B和三个T端点主指标为MAE。Bootstrap在化合物行上重采样，不把feature当成独立重复。

In [ ]:
def metric_record(frame: pd.DataFrame) -> dict:
    kind = frame['kind'].iloc[0]
    y_true = frame['y_true'].to_numpy(dtype=float)
    y_pred = frame['y_prediction'].to_numpy(dtype=float)
    if kind == 'classification':
        prob = frame['y_probability'].to_numpy(dtype=float)
        both = np.unique(y_true).size == 2
        return {
            'n': len(frame), 'n_positive': int(y_true.sum()), 'prevalence': float(y_true.mean()),
            'pr_auc': float(average_precision_score(y_true, prob)) if both else np.nan,
            'roc_auc': float(roc_auc_score(y_true, prob)) if both else np.nan,
            'brier': float(brier_score_loss(y_true, prob)),
            'balanced_accuracy_0.5': float(balanced_accuracy_score(y_true, prob >= 0.5)) if both else np.nan,
            'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'spearman': np.nan,
        }
    rho = spearmanr(y_true, y_pred).statistic if len(frame) >= 3 else np.nan
    return {
        'n': len(frame), 'n_positive': np.nan, 'prevalence': np.nan, 'pr_auc': np.nan, 'roc_auc': np.nan, 'brier': np.nan, 'balanced_accuracy_0.5': np.nan,
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2': float(r2_score(y_true, y_pred)), 'spearman': float(rho),
    }

metric_rows = []
fold_metric_rows = []
for (route, task), frame in oof.groupby(['representation_route', 'task'], sort=False):
    base = {'representation_route': route, 'task': task, 'kind': frame['kind'].iloc[0], 'model': 'nested_selected_primary'}
    metric_rows.append({**base, **metric_record(frame)})
    for fold, fold_frame in frame.groupby('outer_fold'):
        fold_metric_rows.append({**base, 'outer_fold': int(fold), **metric_record(fold_frame)})
metrics = pd.DataFrame(metric_rows)
fold_metrics = pd.DataFrame(fold_metric_rows)

rng = np.random.default_rng(SEED)
bootstrap_rows = []
for (route, task), frame in oof.groupby(['representation_route', 'task'], sort=False):
    kind = frame['kind'].iloc[0]
    values = []
    for _ in range(N_BOOTSTRAP):
        sample = frame.iloc[rng.integers(0, len(frame), size=len(frame))]
        record = metric_record(sample)
        value = record['pr_auc'] if kind == 'classification' else record['mae']
        if np.isfinite(value):
            values.append(value)
    observed_record = metric_record(frame)
    observed = observed_record['pr_auc'] if kind == 'classification' else observed_record['mae']
    bootstrap_rows.append({
        'representation_route': route, 'task': task, 'kind': kind,
        'primary_metric': 'pr_auc' if kind == 'classification' else 'mae',
        'observed': observed, 'ci_low': float(np.quantile(values, 0.025)), 'ci_high': float(np.quantile(values, 0.975)),
        'n_bootstrap_requested': N_BOOTSTRAP, 'n_bootstrap_valid': len(values), 'n_compounds': len(frame),
    })
bootstrap_ci = pd.DataFrame(bootstrap_rows)

calibration_rows = []
p_oof = oof.loc[oof['kind'].eq('classification')].copy()
for route, frame in p_oof.groupby('representation_route'):
    bins = pd.cut(frame['y_probability'], bins=np.linspace(0, 1, 11), include_lowest=True, duplicates='drop')
    for interval, part in frame.groupby(bins, observed=False):
        if len(part):
            calibration_rows.append({
                'representation_route': route, 'probability_bin': str(interval), 'n': len(part),
                'mean_predicted_probability': part['y_probability'].mean(), 'observed_positive_fraction': part['y_true'].mean(),
            })
calibration = pd.DataFrame(calibration_rows)
metrics.to_csv(OUTPUT_DIR / 'sirius_head_metrics.csv', index=False)
fold_metrics.to_csv(OUTPUT_DIR / 'sirius_head_fold_metrics.csv', index=False)
bootstrap_ci.to_csv(OUTPUT_DIR / 'sirius_head_bootstrap_ci.csv', index=False)
calibration.to_csv(OUTPUT_DIR / 'sirius_head_probability_calibration.csv', index=False)
display(metrics)
display(bootstrap_ci)

## 6. 最终QC与可复现运行清单

只有硬性QC全部PASS后，输出才能进入下一阶段的内部模型总比较。覆盖率本身单独报告，不因低于100%而伪造预测。

In [ ]:
qc_records = []
def add_qc(check_id: str, description: str, passed: bool, detail: str = ''):
    qc_records.append({'check_id': check_id, 'description': description, 'status': 'PASS' if passed else 'FAIL', 'detail': detail})

add_qc('SHEAD001', '正负模式absoluteIndex集合完全一致', abs_pos == abs_neg, f'n={len(abs_pos & abs_neg)}')
add_qc('SHEAD002', '指纹维数为5000', n_bits == 5000, f'n_bits={n_bits}')
add_qc('SHEAD003', 'Development化合物ID唯一', not compound_index['inchikey_2d'].duplicated().any(), f'n={len(compound_index)}')
add_qc('SHEAD004', '冻结外折仅为0-4', set(compound_index['cv_fold']) == set(range(5)), str(sorted(compound_index['cv_fold'].unique())))
add_qc('SHEAD005', '所有SIRIUS feature均来自Development', feature_manifest['supervised_compound_role'].eq('development').all())
add_qc('SHEAD006', 'OOF路线-任务-化合物唯一', not oof.duplicated(['representation_route', 'task', 'inchikey_2d']).any(), f'n={len(oof)}')
add_qc('SHEAD007', 'OOF预测值均有限', np.isfinite(oof['y_prediction']).all())
p_values = oof.loc[oof['kind'].eq('classification'), 'y_probability'].to_numpy(dtype=float)
add_qc('SHEAD008', '分类概率均在[0,1]', np.isfinite(p_values).all() and (p_values >= 0).all() and (p_values <= 1).all())
expected_oof = int(coverage.loc[coverage['fold'].eq('all'), 'n_predictable'].sum())
add_qc('SHEAD009', 'OOF行数等于各路线各任务可预测数之和', len(oof) == expected_oof, f'oof={len(oof)}, expected={expected_oof}')
fold_match = oof['cv_fold'].astype(int).eq(oof['outer_fold'].astype(int)).all()
add_qc('SHEAD010', '每条OOF预测来自其冻结外折', fold_match)
all_tasks_routes = set(zip(oof['representation_route'], oof['task']))
expected_tasks_routes = {(route, task) for route in ROUTES_TO_MODEL for task in TASKS}
add_qc('SHEAD011', '所有预定路线和任务均有OOF结果', all_tasks_routes == expected_tasks_routes, f'actual={len(all_tasks_routes)}, expected={len(expected_tasks_routes)}')
model_files = sorted(set(trained_model_paths))
expected_models = len(ROUTES_TO_MODEL) * len(TASKS) * N_OUTER_FOLDS
add_qc('SHEAD012', '所有外折模型均已保存', len(model_files) == expected_models, f'actual={len(model_files)}, expected={expected_models}')

qc = pd.DataFrame(qc_records)
qc.to_csv(OUTPUT_DIR / 'sirius_head_qc_checks.csv', index=False)
display(qc)
if qc['status'].eq('FAIL').any():
    raise RuntimeError('SIRIUS预测头存在硬性QC FAIL，禁止进入下一阶段。')

formal_outputs = [
    OUTPUT_DIR / 'sirius_compound_fingerprint_routes.npz',
    OUTPUT_DIR / 'sirius_compound_fingerprint_index.csv',
    OUTPUT_DIR / 'sirius_feature_to_compound_status.csv.gz',
    OUTPUT_DIR / 'sirius_head_task_coverage.csv',
    OUTPUT_DIR / 'sirius_head_compound_level_oof.csv.gz',
    OUTPUT_DIR / 'sirius_head_selected_hyperparameters.csv',
    OUTPUT_DIR / 'sirius_head_nested_cv_scores.csv.gz',
    OUTPUT_DIR / 'sirius_head_metrics.csv',
    OUTPUT_DIR / 'sirius_head_fold_metrics.csv',
    OUTPUT_DIR / 'sirius_head_bootstrap_ci.csv',
    OUTPUT_DIR / 'sirius_head_probability_calibration.csv',
    OUTPUT_DIR / 'sirius_head_qc_checks.csv',
]
run_manifest = {
    'export_version': 'SIRIUS_PBT_HEADS_V1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'scientific_scope': 'Development public spectra only; ChemVista external evaluation remains locked',
    'quick_test': QUICK_TEST, 'seed': SEED, 'n_outer_folds': N_OUTER_FOLDS, 'n_bootstrap': N_BOOTSTRAP,
    'routes_modelled': ROUTES_TO_MODEL, 'tasks': TASKS,
    'aggregation': {
        'within_mode': 'elementwise median across successful SIRIUS features per compound',
        'cross_mode_alignment': 'absoluteIndex',
        'fusion': 'elementwise mean across available mode-level vectors; no zero imputation',
    },
    'model_selection': 'nested within frozen outer structure-cluster folds; classification PR-AUC; regression negative MAE',
    'software': {
        'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
        'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__,
    },
    'counts': {
        'development_compounds': len(compound_index), 'fingerprint_bits': n_bits, 'oof_rows': len(oof),
        'saved_fold_models': len(model_files),
    },
    'input_hashes': {name: sha256_file(path) for name, path in required_files.items()},
    'output_hashes': {path.name: sha256_file(path) for path in formal_outputs},
    'fold_model_hashes': {path.name: sha256_file(path) for path in sorted(model_files)},
}
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('全部硬性QC通过。')
print('运行清单：', manifest_path)
print('下一步：Notebook 12 对齐SIRIUS、DreaMS、直接谱图、Set Transformer和结构上限的相同化合物OOF结果。')

## 运行完成后的判读顺序

1. 先看`sirius_head_qc_checks.csv`，必须全部PASS。
2. 再看`sirius_head_task_coverage.csv`，这是实际可部署覆盖率。
3. 看`sirius_head_metrics.csv`和`sirius_head_bootstrap_ci.csv`，主融合路线是`available_mode_mean`。
4. 正、负单模式路线用于判断离子模式敏感性，不应只挑表现更好的一条而忽略覆盖率。
5. 本Notebook运行完成后仍不要查看ChemVista性能；应先完成Notebook 12的内部总比较和最终模型冻结。